# Hands-on Exercise 2b — Scaling a Model-Serving Deployment Across Nodes
### AI Operations (AIOps) — Module 3, Lecture 2b | ~75–100 minutes

**Referenced in:** *Module3_Slides_2b_KServe.pptx*

**Objective:** package the RandomForest classifier as a REST prediction service, deploy it on
your 3-node minikube cluster with a `HorizontalPodAutoscaler`, load-test it, and watch
Kubernetes scale replicas up **across multiple nodes** as concurrent traffic increases. Then,
optionally, wrap the same container in a real KServe `InferenceService`.

**This exercise is tiered:**
- **Tier 1 (everyone completes this):** plain Kubernetes `Deployment` + `Service` + `HPA`. This
  is exactly what KServe generates and manages for you under the hood — writing it out by hand
  first is what makes Tier 2 feel like "the same thing, automated" instead of magic. Guaranteed
  to work with nothing beyond `kubectl` and your existing minikube cluster.
- **Tier 2 (optional extension):** install real KServe in *RawDeployment* mode and deploy an
  `InferenceService` wrapping the same container. RawDeployment avoids the Istio/Knative
  dependency chain that makes full KServe installs fragile on a laptop-scale cluster — the
  tradeoff is that RawDeployment does **not** support true scale-to-zero (that needs the
  Knative-based serverless mode, covered conceptually in the slide deck).

**Prerequisites:** completion of Lecture 2a (a running 3-node minikube cluster). Enable the
metrics-server addon, which the HPA needs to read CPU usage:
```bash
minikube addons enable metrics-server
```

**Deliverable:**
- The predictor running locally and responding correctly to a test request
- `kubectl get hpa -w` output showing replica count rising under load
- `kubectl get pods -o wide` output showing replicas spread across multiple nodes
- The load-test summary table and a plot of latency/replica-count vs. concurrency
- (Optional, Tier 2) the same experiment repeated against a real KServe `InferenceService`

## Part 0 — Start a three-node cluster

We start small (3 nodes, 4 CPUs) to demonstrate multi-**pod** parallelism on a single node first,
before scaling out to multiple nodes.

In [ ]:
# Run this in a TERMINAL (not this notebook cell) -- it takes 1-3 minutes:
#
#   minikube start --nodes 3 --cpus 4 --memory 4096 --driver=docker
#
# Then verify from here:
import subprocess

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

sh("kubectl cluster-info")
sh("kubectl get nodes -o wide")

_Expected output:_ one node, `STATUS: Ready`. If `kubectl cluster-info` fails, minikube likely
isn't running yet — go back to the terminal and confirm `minikube start` finished successfully.

## Part 1 — Train and export the model

Same dataset as Lecture 2a — deliberate continuity. We use the best hyperparameters a student would have found in that exercise's grid search.

In [ ]:
import subprocess, os

# Reuse Lecture 2a's dataset generator if you still have it; otherwise regenerate here.
if not os.path.exists("generate_dataset.py"):
    with open("generate_dataset.py", "w") as f:
        f.write('''
import argparse
import pandas as pd
from sklearn.datasets import make_classification

def generate(n_samples=6000, n_features=20, n_informative=12, n_classes=2, random_state=42):
    X, y = make_classification(n_samples=n_samples, n_features=n_features,
        n_informative=n_informative, n_redundant=4, n_classes=n_classes,
        class_sep=1.1, random_state=random_state)
    columns = [f"feature_{i:02d}" for i in range(n_features)]
    df = pd.DataFrame(X, columns=columns)
    df["label"] = y
    return df

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", default="ml_job_dataset.csv")
    parser.add_argument("--n-samples", type=int, default=60000)
    parser.add_argument("--n-features", type=int, default=20)
    args = parser.parse_args()
    generate(n_samples=args.n_samples, n_features=args.n_features).to_csv(args.out, index=False)
''')

os.makedirs("data", exist_ok=True)
subprocess.run(["python3", "generate_dataset.py", "--out", "data/ml_job_dataset.csv"], check=True)
print("Dataset ready.")

In [ ]:
%%writefile train_and_export_model.py
"""
train_and_export_model.py — AI Operations (AIOps), Module 3 Lecture 2b
Trains and exports the model served by predictor_app.py.
"""
import argparse
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/ml_job_dataset.csv")
    parser.add_argument("--out", default="data/model.joblib")
    parser.add_argument("--n-estimators", type=int, default=200)
    parser.add_argument("--max-depth", type=int, default=12)
    args = parser.parse_args()

    df = pd.read_csv(args.data)
    X = df.drop(columns=["label"])
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    model = RandomForestClassifier(n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42, n_jobs=1)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"accuracy={accuracy_score(y_test, preds):.4f}  f1={f1_score(y_test, preds):.4f}")

    joblib.dump({"model": model, "feature_columns": list(X.columns)}, args.out)
    print(f"Saved model bundle to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
subprocess.run(["python3", "train_and_export_model.py"], check=True)
print(os.path.getsize("data/model.joblib"), "bytes")

## Part 2 — Write the predictor app

A small FastAPI app implementing KServe's V1 inference protocol (`POST /v1/models/{name}:predict`) — the same request/response shape a real InferenceService expects, so Tier 1 and Tier 2 stay compatible.

In [ ]:
%%writefile predictor_app.py
"""
predictor_app.py — AI Operations (AIOps), Module 3 Lecture 2b
FastAPI predictor implementing the KServe V1 inference protocol.
"""
import os, socket, time
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH = os.environ.get("MODEL_PATH", "data/model.joblib")
POD_NAME = os.environ.get("POD_NAME", socket.gethostname())
NODE_NAME = os.environ.get("NODE_NAME", "unknown")

app = FastAPI(title="RF Classifier Predictor")
_bundle = None


@app.on_event("startup")
def load_model():
    global _bundle
    _bundle = joblib.load(MODEL_PATH)
    print(f"Loaded model ({len(_bundle['feature_columns'])} features) on pod={POD_NAME} node={NODE_NAME}")


class PredictRequest(BaseModel):
    instances: list[list[float]]


@app.get("/healthz")
def healthz():
    return {"status": "ok", "pod": POD_NAME, "node": NODE_NAME}


@app.get("/v1/models/{model_name}")
def model_ready(model_name: str):
    if _bundle is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")
    return {"name": model_name, "ready": True}


@app.post("/v1/models/{model_name}:predict")
def predict(model_name: str, request: PredictRequest):
    if _bundle is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")
    model = _bundle["model"]
    expected = len(_bundle["feature_columns"])
    for row in request.instances:
        if len(row) != expected:
            raise HTTPException(status_code=400, detail=f"Expected {expected} features, got {len(row)}")

    t0 = time.time()
    predictions = model.predict(request.instances).tolist()
    latency_ms = round((time.time() - t0) * 1000, 2)

    return {
        "predictions": predictions,
        "served_by_pod": POD_NAME,
        "served_by_node": NODE_NAME,
        "latency_ms": latency_ms,
    }


**Sanity-check the predictor locally** before containerizing it — start it in the background and hit it with a real request:

In [ ]:
import subprocess, time, requests, os

env = os.environ.copy()
env.update({"MODEL_PATH": "data/model.joblib", "POD_NAME": "local-test", "NODE_NAME": "local"})

proc = subprocess.Popen(
    ["uvicorn", "predictor_app:app", "--host", "0.0.0.0", "--port", "28080"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
#time.sleep(3)

In [ ]:
health = requests.get("http://localhost:28080/healthz").json()
print("Health check:", health)

payload = {"instances": [[0.5] * 20]}
pred = requests.post("http://localhost:28080/v1/models/rf-classifier:predict", json=payload).json()
print("Prediction:", pred)

In [ ]:
# Stop the local test server -- we'll run the real thing inside Kubernetes from here on.
proc.terminate()
proc.wait(timeout=5)
print("Local test server stopped.")

## Part 3 — Containerize the predictor

In [ ]:
%%writefile requirements-predictor.txt
fastapi
uvicorn[standard]
scikit-learn
joblib
pydantic


In [ ]:
%%writefile Dockerfile.predictor

# Stage 1: Build dependencies
FROM python:3.11-slim AS builder

WORKDIR /app

RUN python -m venv /opt/venv
# Ensure subsequent commands use the virtualenv
ENV PATH="/opt/venv/bin:$PATH"

COPY requirements-predictor.txt .
RUN pip install --no-cache-dir -U pip
RUN pip install --no-cache-dir -r requirements-predictor.txt

# Stage 2: Final runtime
FROM python:3.11-slim AS runner

WORKDIR /app

# Copy the virtual environment from the builder stage
COPY --from=builder /opt/venv /opt/venv
COPY predictor_app.py .
COPY data/model.joblib ./model.joblib
ENV MODEL_PATH=/app/model.joblib

# Critical: Update the PATH so the system can find uvicorn
ENV PATH="/opt/venv/bin:$PATH"

EXPOSE 8080
CMD ["uvicorn", "predictor_app:app", "--host", "0.0.0.0", "--port", "8080"]

Build directly into minikube's image store and load it onto every node (same two-step
pattern as Lecture 2a — build once, then make sure ALL nodes actually have it):

```bash
minikube image build -t rf-predictor:latest -f Dockerfile.predictor --all . 
```

OR

```bash
docker build -t rf-predictor:latest -f Dockerfile.predictor .
minikube image load rf-predictor:latest
```

## Part 4 (Tier 1) — Deploy with a plain Kubernetes Deployment + Service + HPA

This is guaranteed to work with nothing beyond `kubectl` and your existing 3-node cluster.

In [ ]:
%%writefile deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: rf-predictor
  labels:
    app: rf-predictor
spec:
  replicas: 1
  selector:
    matchLabels:
      app: rf-predictor
  template:
    metadata:
      labels:
        app: rf-predictor
    spec:
      containers:
      - name: predictor
        image: rf-predictor:latest
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 8080
        env:
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        resources:
          requests:
            cpu: "250m"
            memory: "256Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"
        readinessProbe:
          httpGet:
            path: /healthz
            port: 8080
          initialDelaySeconds: 3
          periodSeconds: 5
        livenessProbe:
          httpGet:
            path: /healthz
            port: 8080
          initialDelaySeconds: 5
          periodSeconds: 10


In [ ]:
%%writefile service.yaml
apiVersion: v1
kind: Service
metadata:
  name: rf-predictor-svc
spec:
  type: NodePort
  selector:
    app: rf-predictor
  ports:
  - port: 80
    targetPort: 8080
    nodePort: 30080


In [ ]:
%%writefile hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: rf-predictor-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: rf-predictor
  minReplicas: 1
  maxReplicas: 6
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 50
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 60
    scaleUp:
      stabilizationWindowSeconds: 0
      policies:
      - type: Pods
        value: 2
        periodSeconds: 15


Apply all three and verify, in a terminal:

```bash
minikube addons enable metrics-server   # if not already enabled in Lecture 2a
kubectl apply -f deployment.yaml
kubectl apply -f service.yaml
kubectl apply -f hpa.yaml

kubectl get pods -o wide
kubectl get hpa
```

Wait ~1 minute for `metrics-server` to report real CPU numbers (`kubectl get hpa` will show
`<unknown>` under `TARGETS` until then).

## Part 5 (Tier 1) — Load test and watch it scale

Find the URL to hit — minikube's `service` command resolves the NodePort to a reachable address
for you:

```bash
minikube service rf-predictor-svc --url
```

Use that URL below (it will look like `http://192.168.49.2:30080`).

In [ ]:
%%writefile load_test.py
"""
load_test.py — AI Operations (AIOps), Module 3 Lecture 2b
Fires concurrent prediction requests and records latency + which pod/node served each one.
"""
import argparse, concurrent.futures, random, time
import pandas as pd
import requests


def make_random_instance(n_features=20):
    return [random.uniform(-3, 3) for _ in range(n_features)]


def fire_one_request(url, n_features, timeout=10):
    payload = {"instances": [make_random_instance(n_features)]}
    t0 = time.time()
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
        latency_ms = (time.time() - t0) * 1000
        if resp.status_code == 200:
            body = resp.json()
            return {"status_code": 200, "latency_ms": round(latency_ms, 2),
                     "served_by_pod": body.get("served_by_pod"), "served_by_node": body.get("served_by_node")}
        return {"status_code": resp.status_code, "latency_ms": round(latency_ms, 2),
                 "served_by_pod": None, "served_by_node": None}
    except requests.exceptions.RequestException as e:
        return {"status_code": -1, "latency_ms": (time.time() - t0) * 1000,
                 "served_by_pod": None, "served_by_node": None, "error": str(e)}


def run_at_concurrency(url, concurrency, n_requests, n_features=20):
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(fire_one_request, url, n_features) for _ in range(n_requests)]
        for f in concurrent.futures.as_completed(futures):
            results.append(f.result())
    return pd.DataFrame(results)


def run_sweep(url, concurrency_levels, requests_per_level, n_features=20):
    all_rows = []
    for c in concurrency_levels:
        print(f"--- concurrency={c}, {requests_per_level} requests ---")
        t0 = time.time()
        df = run_at_concurrency(url, c, requests_per_level, n_features)
        wall_seconds = time.time() - t0
        df["concurrency"] = c
        success = (df["status_code"] == 200).sum()
        p50 = df.loc[df["status_code"] == 200, "latency_ms"].median()
        print(f"  success={success}/{requests_per_level}  p50={p50:.1f}ms  "
              f"throughput={success/wall_seconds:.1f} req/s")
        all_rows.append(df)
    return pd.concat(all_rows, ignore_index=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--url", required=True)
    parser.add_argument("--concurrency-levels", type=int, nargs="+", default=[1, 5, 20])
    parser.add_argument("--requests-per-level", type=int, default=60)
    parser.add_argument("--out-csv", default="load_test_results.csv")
    args = parser.parse_args()

    df = run_sweep(args.url, args.concurrency_levels, args.requests_per_level)
    df.to_csv(args.out_csv, index=False)


Run the sweep from a terminal (so you can watch `kubectl get pods -o wide -w` and
`kubectl get hpa -w` live in two other terminals at the same time):

```bash
python3 load_test.py --url http://192.168.49.2:30080/v1/models/rf-classifier:predict \
    --concurrency-levels 1 10 40 --requests-per-level 1000
```

Then load the results back here for analysis:

In [ ]:
import pandas as pd

df = pd.read_csv("load_test_results.csv")
summary = (
    df[df["status_code"] == 200]
    .groupby("concurrency")
    .agg(
        requests=("status_code", "count"),
        distinct_pods=("served_by_pod", "nunique"),
        distinct_nodes=("served_by_node", "nunique"),
        p50_latency_ms=("latency_ms", "median"),
        p95_latency_ms=("latency_ms", lambda s: s.quantile(0.95)),
    )
    .reset_index()
)
summary

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(summary["concurrency"], summary["p50_latency_ms"], marker="o", label="p50")
ax1.plot(summary["concurrency"], summary["p95_latency_ms"], marker="s", label="p95")
ax1.set_xlabel("Concurrency")
ax1.set_ylabel("Latency (ms)")
ax1.set_title("Latency vs. Concurrency")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(summary["concurrency"], summary["distinct_pods"], marker="o", color="tab:green", label="Distinct pods")
ax2.plot(summary["concurrency"], summary["distinct_nodes"], marker="s", color="tab:red", label="Distinct nodes")
ax2.set_xlabel("Concurrency")
ax2.set_ylabel("Count")
ax2.set_title("Replica / Node Spread vs. Concurrency")
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig("load_test_plot.png", dpi=120)
print("Saved load_test_plot.png")

_Discussion:_ as concurrency rises, does `distinct_pods` and `distinct_nodes` increase? Cross-check
against `kubectl get hpa` history — did the HPA's `REPLICAS` column rise at roughly the same
point p95 latency started climbing? That correlation IS the autoscaler working as designed:
scale out before latency becomes unacceptable, not after.

## Part 6 (Tier 2, Optional) — Wrap the Same Container in a Real KServe InferenceService

This section installs KServe itself. It's optional: Tier 1 already fully demonstrates
multi-node scaling. Do this if time and cluster resources allow.

### Install KServe (RawDeployment mode — no Istio/Knative required)

```bash
# Install cert-manager (a KServe dependency)
kubectl apply -f https://github.com/cert-manager/cert-manager/releases/latest/download/cert-manager.yaml
kubectl wait --for=condition=Available --timeout=120s deployment --all -n cert-manager

# Install KServe with RawDeployment as the default deployment mode
curl -s "https://raw.githubusercontent.com/kserve/kserve/master/hack/quick_install.sh" | bash -s -- -r
```

OR

```bash
# Standard mode
curl -sL "https://github.com/kserve/kserve/releases/download/v0.20.0/kserve-standard-mode-full-install-with-manifests.sh" | bash
```

> **Version note:** KServe's install script and flags change between releases. Verify the exact
> command against the current KServe Quickstart guide (kserve.github.io/website) before class —
> the `-r` (raw deployment) flag is the important part to preserve if the script name/URL changes.

In [ ]:
%%writefile inference-service.yaml
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: rf-classifier
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
    serving.kserve.io/autoscalerClass: hpa
    serving.kserve.io/metrics: cpu
    serving.kserve.io/targetUtilizationPercentage: "50"
spec:
  predictor:
    minReplicas: 1
    maxReplicas: 6
    containers:
    - name: kserve-container
      image: rf-predictor:latest
      imagePullPolicy: IfNotPresent
      ports:
      - containerPort: 8080
      env:
      - name: POD_NAME
        valueFrom:
          fieldRef:
            fieldPath: metadata.name
      - name: NODE_NAME
        valueFrom:
          fieldRef:
            fieldPath: spec.nodeName
      resources:
        requests:
          cpu: "250m"
          memory: "256Mi"
        limits:
          cpu: "500m"
          memory: "512Mi"


Apply it and inspect:

```bash
kubectl apply -f inference-service.yaml
kubectl get inferenceservice rf-classifier
kubectl get pods -l serving.kserve.io/inferenceservice=rf-classifier -o wide
```

Notice this created the SAME kind of Deployment/Service/HPA trio as Tier 1 — just generated and
managed for you. Re-run `load_test.py` against this new endpoint and confirm you see the same
scaling behaviour.

**Connecting back to Module 1:** in production, `spec.predictor.model.storageUri` would point at
your MLflow Model Registry (`models:/rf-classifier/Production`) instead of a custom container
with a baked-in model file — we used a custom container here purely because minikube's default
storage provisioner is node-local, which would break multi-node access to a shared model file.
With real networked/object storage in production, the storageUri pattern from the original
lecture slides is what you'd actually use.

## ✅ Deliverable Checklist
- [ ] Predictor tested locally (health check + prediction) before containerizing
- [ ] Tier 1: Deployment + Service + HPA applied successfully; `kubectl get hpa` shows real CPU metrics
- [ ] Load test run at ≥ 3 concurrency levels; `distinct_nodes` > 1 at the highest concurrency level
- [ ] Latency and replica/node-spread plots
- [ ] A short discussion connecting rising concurrency, rising latency, and rising replica count
- [ ] (Optional) Tier 2: KServe installed and an InferenceService deployed and load-tested

*This completes the Module 3, Lecture 2 hands-on exercise sequence.*